# 系统提示词

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
  model='deepseek-chat',
  system_prompt='你以海盗的口吻来回答用户问题'
)

stream = agent.stream(
  {'messages': [HumanMessage('你是加勒比海盗吗？')]},
  stream_mode='messages'
)

for chunk,metadata in stream:
  print(chunk.content, end='', flush=True)

# 提示词工程
所谓提示词工程，就是通过优化提示词使模型输出的结果更符合业务需要的过程——是个不断优化的过程

一般来说，系统提示词会包含以下几个部分，通常按此顺序排列：
- 身份角色： 描述AI的职责，沟通风格和总体目标。
- 指令说明： 指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及绝对不能做什么？
- 对话示例： 提供可能的输入示例，以及模型期望的输出。
- 背景信息： 向模型提供生成响应所需的额外信息，例如RAG的额外知识库数据。

## 角色+指令更能明确提示词

In [ ]:
system_prompt='''
  # 角色
  - 你是一个编程助手，你帮助用户编写Python代码。

  # 指令
  - 定义变量时，使用snake case命名法，而不是camel case命名法。
  - 不要返回markdown格式说明，仅仅返回代码即可。
'''

agent = create_agent(
  model='deepseek-chat',
  system_prompt=system_prompt
)

stream = agent.stream(
  {'messages': [HumanMessage('怎样定义string变量记录学校名字')]},
  stream_mode='messages'
)

for chunk,metadata in stream:
  print(chunk.content, end='', flush=True)

## 对话示例
Few-shot是为模型提供多种示例的方法，以便它可以学习行为模式并生成更准确的响应。

In [ ]:
system_prompt = '''
  # 身份
  - 你是一个科幻作家，根据用户的要求创建一个太空之都。
  # 示例
  user: 月球的首都是什么？
  assistant: 月华城(Lunara) —— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

  user:火星的首都是什么?
  assistant:赤晶城(Aresia)一 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
'''

# 创建智能体
agent = create_agent(
  model='deepseek-chat',
  system_prompt=system_prompt
)

stream = agent.stream(
  {'messages': [HumanMessage('金星的首都是什么？')]},
  stream_mode='messages'
)

for chunk,metadata in stream:
  print(chunk.content, end='', flush=True)

## 结构化输出
### a. 基于提示词的结构优化

In [ ]:
system_prompt = '''
  # 身份
  - 你是一个科幻作家，根据用户的要求创建一个太空之都。
  # 指令
  - 请务必以JSON格式输出，不要加任何markdown样式。
  # 示例
  user: 月球的首都是什么？
  assistant:
  {
    "name":"月华市(Lunaria)",
    "vibe":"冷冽、高效、革新"，
    "location":"位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成"，
    "economy":"氮-3能源开采、量子通信枢纽、尖端生物圈农业"
  }
'''

agent = create_agent(
  model='deepseek-chat',
  system_prompt=system_prompt
)

response = agent.invoke(
  {'messages': [HumanMessage('金星的首都是什么？')]}
)

print(response['messages'][-1].content)

### b.基于Model的结构化输出
在LangChain中会更简单，仅需两步即可：
1. 定义一个数据类型(基于pydantic)
2. 创建智能体，设置输出格式

In [ ]:
from pydantic import BaseModel
# 首先定义一个类，用于封装模型要输出的数据
class CaptitalModel(BaseModel):
  name: str
  location: str
  vibe: str
  economy: str

agent = create_agent(
  model='deepseek-chat',
  system_prompt='你是一个科幻作家，根据用户的要求创建一个太空之都。',
  response_format=CaptitalModel  # 设置结构化输出的格式
)

response = agent.invoke(
  {'messages': [HumanMessage('月球的首都是什么？')]}
)

print(response)

In [12]:
city = response['structured_response']
# print(city)
print(f'{city.name}——{city.location}——{city.vibe}——{city.economy}')

静海城（Serenity）——月球正面赤道附近的静海（Mare Tranquillitatis）西缘，位于一座被改造过的熔岩管穹顶之下，地表之上是密集的太阳能阵列与起降场，地下三十层则是城市的真正主体。——冷静、秩序、带有一种克制的庄重感。城市没有天空，取而代之的是穹顶投影出的"人造天光"，按照二十四小时周期由冷白渐变为暖金。街道以低重力步道相连，行人轻盈地滑行；空气循环系统发出恒定的低鸣，被居民称作"月亮的呼吸"。这里既有地球移民的乡愁，也有月生一代的疏离——他们从未见过真正的雨。——以氦-3开采与出口为支柱，向地球及地月轨道站供应聚变燃料；同时是地月贸易的中枢港，月球的水冰加工、稀土冶炼与真空精密制造业在此集散。城中还设有"联合月区议会"与月面交通管制总署，掌握着整个月球的航运与能源调度权。
